# Experiment 1: Simulation environment

This notebook is used to experiment with the simulation environment. Add new few features, check if it's working properly, examples, etc.

## Initialize

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [2]:
from math import radians, pi, sin
import matplotlib.pyplot as plt
from tqdm.auto import trange
import xarray.ufuncs as xrf
import numba as nb
import numpy as np
from numba.experimental import jitclass
from IPython.display import display, JSON

from cw.context import time_it
from cw.filters import smooth_signal

from traj1.logger import Logger

from environment import Environment, Stage, AP_NONE, AP_PITCH_CONTROL, AP_FLIGHT_PATH_CONTROL, AP_PITCH_RATE_CONTROL

## Create new Environment

Creates a new environment with an attached logger.

In [12]:
env = Environment(
    dt=0.01,
    surface_diameter=1737.4e3,
    mu=4.9048695e12,
    stages=(
        Stage(
            dry_mass=1,
            propellant_mass=0.5,
            specific_impulse=80,
            thrust=2*1.7),
    ),
    initial_altitude=0,
    initial_theta_e=radians(90),
    initial_longitude=radians(90),
    gamma_controller_gains=(4, 0, 0.2),
    theta_controller_gains=(10, 0, 0.0),
    controller_theta_dot_limits=(-1, 1),
    autopilot_mode=AP_PITCH_CONTROL
)


In [13]:
logger = Logger()
logger.register_time_attribute(env.sim, "t")
logger.register(env.sim, "env", [
    "h",
    "gamma_e", "theta_e", "theta_i_dot",
    "ap_comm_gamma_e", "ap_comm_theta_e",
    "action_autopilot_mode", "action_autopilot_reference", "vii", "xii", "fii_thrust", "mass", "mass_dot"])

In [35]:
n_episodes = 1
max_time = 300

last_episode_result = None
batch_results = None

logger.episode_reset()
logger.batch_reset()
with time_it("sim"):
    for episiode_idx in range(n_episodes):
        observation = env.reset()
        for i in range(int(max_time / env.sim.integrator.h)):
            pitch_angle = 0.5 * pi + 0.5 * sin(env.sim.t)
            observation, reward, done, info = env.step(pitch_angle)
            logger.step()
            if done:
                continue
        last_episode_result = logger.episode_finish()
batch_results = logger.batch_finish()

sim: 1.5286625499993534 [s]


In [36]:
last_episode_result

<xarray.Dataset>
Dimensions:                         (d_2_0: 2, t: 30000)
Coordinates:
  * t                               (t) float64 0.01 0.02 0.03 ... 300.0 300.0
Dimensions without coordinates: d_2_0
Data variables: (12/13)
    env_h                           (t) float64 0.0 -0.0001625 ... -1.585e+03
    env_gamma_e                     (t) float64 -1.571 -1.571 ... -1.582 -1.582
    env_theta_e                     (t) float64 1.571 1.571 ... 1.078 1.078
    env_theta_i_dot                 (t) float64 0.0 0.05 ... -0.07029 -0.06537
    env_ap_comm_gamma_e             (t) float64 nan nan nan nan ... nan nan nan
    env_ap_comm_theta_e             (t) float64 1.571 1.576 ... 1.071 1.071
    ...                              ...
    env_action_autopilot_reference  (t) float64 1.571 1.576 ... 1.071 1.071
    env_vii                         (t, d_2_0) float64 -9.95e-19 ... -186.2
    env_xii                         (t, d_2_0) float64 1.064e-10 ... 1.736e+06
    env_fii_thrust                  (t, d_2_0) float64 2.082e-16 3.4 ... 0.0 0.0
    env_mass                        (t) float64 1.5 1.5 1.5 1.5 ... 1.0 1.0 1.0
    env_mass_dot                    (t) float64 -0.004332 -0.004332 ... 0.0 0.0

In [37]:
plt.figure()
last_episode_result.env_h.plot.line(x="t")

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [38]:
plt.figure()
plt.plot(last_episode_result.env_xii.values[:, 0], last_episode_result.env_xii.values[:, 1])

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [140]:
env.reset()

In [498]:
a = pi * 0.5

env.sim.step((
    True,
    False,
    nb.int32(1),
    nb.float64(pi * 0.4),
))
env.sim_states_dict()

{'t': '4.959999999999939',
 'action_engine_on': 'True',
 'action_drop_stage': 'False',
 'action_autopilot_mode': '1',
 'action_autopilot_reference': '1.2566370614359172',
 'ap_comm_gamma_e': '1.2566370614359172',
 'ap_comm_theta_e': '1.6343987944770575',
 'gii': '[-2.51557786e-06 -1.62489013e+00]',
 'xii': '[2.68977215e+00 1.73740765e+06]',
 'vii': '[1.32699945 3.1147535 ]',
 'aii': '[0.05744953 0.67393445]',
 'tei': '[[-1.00000000e+00  1.54815259e-06]\n [ 1.54815259e-06  1.00000000e+00]]',
 'vie': '[-1.32699463  3.11475555]',
 'fii_thrust': '[0.08494602 3.39893868]',
 'theta_i': '1.545809604739566',
 'theta_i_dot': '0.08858764158489985',
 'theta_e': '1.5458111528921576',
 'mass': '1.47855504587152',
 'mass_dot': '-0.004332313965341487',
 'h': '7.646192363463342',
 'engine_on': 'True',
 'stage_state': '1',
 'stage_idx': '0',
 'stage_ignitions_left': '-1',
 'gamma_i': '1.1680478716984262',
 'gamma_e': '1.1680494198510174',
 'longitude': '1.570794778642305',
 'reward': '0.0',
 'score': '

{'t': '0.24000000000000007',
 'action_engine_on': 'True',
 'action_drop_stage': 'False',
 'action_autopilot_mode': '1',
 'action_autopilot_reference': '1.5707963267948966',
 'ap_comm_gamma_e': 'nan',
 'ap_comm_theta_e': 'nan',
 'gii': '[-9.94966983e-17 -1.62490440e+00]',
 'xii': '[1.06385069e-10 1.73740002e+06]',
 'vii': '[9.44265504e-18 1.54210260e-01]',
 'aii': '[3.93928797e-17 6.43334547e-01]',
 'tei': '[[-1.000000e+00  6.123234e-17]\n [ 6.123234e-17  1.000000e+00]]',
 'vie': '[-3.08148791e-33  1.54210260e-01]',
 'fii_thrust': '[2.08189956e-16 3.40000000e+00]',
 'theta_i': '1.5707963267948966',
 'theta_i_dot': '0.0',
 'theta_e': '1.5707963267948966',
 'mass': '1.4989602446483161',
 'mass_dot': '-0.004332313965341487',
 'h': '0.018369183409959078',
 'engine_on': 'True',
 'stage_state': '1',
 'stage_idx': '0',
 'stage_ignitions_left': '-3',
 'gamma_i': '1.5707963267948966',
 'gamma_e': '1.5707963267948966',
 'longitude': '1.5707963267948966',
 'reward': '0',
 'score': '0.0',
 'done': 

In [8]:
spec = [
    ("bar", nb.boolean),
    ("bas", nb.boolean),
    ("qux", nb.int32),
    ("qqq", nb.float64),
]
@jitclass(spec)
class Foo:
    def __init__(self):
        self.bar = 0
        self.bas = 0
        self.qux = 0
        self.qqq = 0
        
    def set(self, new_value):
        self.bar, self.bas, self.qux, self.qqq = new_value

In [9]:
foo = Foo()

foo.set((
    True,
    False,
    nb.int32(1),
    nb.float64(pi * 0.5)
))

In [10]:
foo.qqq

1.5707963267948966